In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""Private Markets and Real Estate

Attributes
----------
private markets
real estate
cash flows
quotes
transactions
excel
pdf
lookthrough
"""

toggle_code("Toggle Docstring")

# Private Markets and Real Estate

Private markets and real estate notebook containing examples for modelling cash flows and portfolio securitization, among other things.

The following list provides a step-by-step overview of the sections that will be covered. More detail on each section will be provided as we move through this notebook.

Load data
1. **[Setup LUSID](#1.-Setup-LUSID)**
    - Packages
2. **[Properties & Portfolios](#2.-Properties-&-Portfolios)**
3. **[Securitise Porfolios](#3.-Securitise-portfolios)**
4. **[Book in Cash](#4.-Book-in-Cash)**
    - Capital Commitments
    - Capital Calls
    - Expenses
5. **[Load Data into Portfolios](#5.-Load-Data-into-Portfolios)**
    - Private Equity Fund Instruments
    - Real Estate Fund Instruments
    - Private Debt Fund Instruments
    - Transactions
    - Quotes
    - Recipes
8. **[Excel Report Writer](#6.-Excel-ILPA-Report-Writer-(for-General-Partnership))** (for General Partnership)
    - Data Consolidation
    - Excel Output
6. **[Real Estate Cash Flows](#7.-Real-Estate-Cash-Flows)**
    - Override Cash Flows
    - Book Bond
    - Data Calculations
    - Data Visualisation
7. **[PDF Report Writer](#8.-PDF-Report-Writer-(for-Real-Estate-Fund))** (for Real Estate Fund)

In [ ]:
%pip install fpdf

# 1. Setup LUSID

Import modules (both LUSID and non-LUSID) and define our API factories.

In [ ]:
# Import generic non-LUSID packages
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import pytz
import time
from IPython.core.display import HTML
import openpyxl
import matplotlib.pyplot as plt
import matplotlib as mpl
import random
from dateutil.relativedelta import relativedelta
from fpdf import FPDF
from IPython.display import IFrame

# Import key modules from the LUSID package
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as lm
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException

# Import key functions from Lusid-Python-Tools and other packages
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.cocoon import load_from_data_frame
from finbourne_sdk_utils.cocoon.cocoon_printer import (
    format_instruments_response,
    format_holdings_response,
    format_portfolios_response,
    format_transactions_response,
    format_quotes_response,)
from finbourne_sdk_utils.cocoon.transaction_type_upload import upsert_transaction_type_alias
from finbourne_sdk_utils.lpt.lpt import to_date
import finbourne.sdk.services.drive as ld


# Set DataFrame display formats
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

# Set the secrets path
secrets_path = os.getenv("FBN_SECRETS_PATH")

# For running the notebook locally
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Authenticate our user and create our API client
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)

print("LUSID Environment Initialised")
print(
    "LUSID API Version :",
    api_factory.build(lu.ApplicationMetadataApi).get_lusid_versions().build_version,
)

In [ ]:
# LUSID Variable Definitions
transaction_portfolios_api = api_factory.build(lu.TransactionPortfoliosApi)
instruments_api = api_factory.build(lu.InstrumentsApi)
property_definitions_api = api_factory.build(lu.PropertyDefinitionsApi)
configuration_recipe_api = api_factory.build(lu.ConfigurationRecipeApi)
structured_result_data_api = api_factory.build(lu.StructuredResultDataApi)
aggregation_api = api_factory.build(lu.AggregationApi)
transaction_config_api = api_factory.build(lu.TransactionConfigurationApi)

In [ ]:
scope = "privateMarkets"

# 2. Properties & Portfolios

First, we need to instantiate our properties and portfolios.

The properties we create are:
1. **Investor**: Investor who is responsible for a transaction
2. **Holding Class**:  Class that the holding can be categorised into
3. **Expense Type**: Category of an expense (only for expense transactions)

We also create 4 portfolios:
1. **General Partnership**
2. **Real Estate Fund**
3. **Private Equity Fund**
4. **Private Debt Fund**

The following diagram illustrates the structure of funds that we are building out, with each box representing a portfolio.

We are able to associate the Real Estate, Private Equity and Private Debt fund portfolios with the General Partnership fund through a process called 'securitisation'. We will cover this in more detail during the next section.

![Image showing fund structure of this notebook](./data/fund_structure.png "Fund Structure")

In [ ]:
# Define custom property code and display names
property_args = {
    "expense_type": "Expense Type",
    "investor": "Investor",
    "holding_class": "Holding Class"
}

# Define function to upsert properties
def transactionPropertyUpsert(k, v):
    try:
        resp = property_definitions_api.create_property_definition(
            create_property_definition_request=lm.CreatePropertyDefinitionRequest(
                domain="Transaction",
                scope=scope,
                code=k,
                value_required=None,
                display_name=v,
                data_type_id=lm.ResourceId(scope="system", code="string"),
                life_time=None,
            )
        )
        print(f"{resp.key} property created")

    except ApiException as e:
        if json.loads(e.body)["code"] == 124:  # PropertyAlreadyExists
            print(json.loads(e.body)["title"])
        else:
            raise e


# Loop through dictionary and call function
for k, v in property_args.items():
    transactionPropertyUpsert(k, v)

In [ ]:
# Details of the portfolios we want to create, name and SHKs
portfolio_details = {
    "generalPartnership": [f"Transaction/{scope}/investor", f"Transaction/{scope}/holding_class"],
    "realEstateFund": [f"Transaction/{scope}/holding_class"],
    "privateEquityFund": [],
    "privateDebtFund": []
}


def transactionPortfolioUpsert(portfolio_name, sub_holding_keys):
    try:
        resp = transaction_portfolios_api.create_portfolio(
            scope=scope,
            create_transaction_portfolio_request=lm.CreateTransactionPortfolioRequest(
                display_name=portfolio_name,
                code=portfolio_name,
                base_currency="GBP",
                created="2021-01-01T00:00:00+00:00",
                sub_holding_keys=sub_holding_keys,
            ),
        )
        print(f"{resp.display_name} portfolio created")

    except ApiException as e:
        if json.loads(e.body)["code"] == 112:  # PortfolioWithIdAlreadyExists
            print(json.loads(e.body)["title"])
        else:
            raise e
        
for portfolio_name, sub_holdings_keys in portfolio_details.items():
    transactionPortfolioUpsert(portfolio_name, sub_holdings_keys)

# 3. Securitise Portfolios

The portfolios we have just created will now undergo a process known as 'securitisation'. By doing this, we encapsulate a portfolio into a security which then allows us to book it into a parent portfolio in the same way we would for any traditional security. In this case, we are securitising the child funds and then booking them into our General Partnership portfolio, creating a fund-of-funds architecture. 

The diagram below shows the fund structure.

![Image showing portfolio securitisation](./data/lookthrough.png "Securites Portfolios")

In [ ]:
# Create a list of securitised portfolios and IDs, using our dictionary as basis
portfolio_details.pop("generalPartnership")
securitised_portfolio_names = list(portfolio_details.keys())
securitised_portfolio_ids = [portfolio_name + "Securitised" for portfolio_name in securitised_portfolio_names] 
securitised_portfolios = list(zip(securitised_portfolio_names, securitised_portfolio_ids))

securitised_portfolios

In [ ]:
# Insert the new portfolio instruments into LUSID
for portfolio, instrument_id in securitised_portfolios:
    try:
        instr_result = instruments_api.upsert_instruments(
            request_body={
                "look_through": lm.InstrumentDefinition(
                    name=portfolio,
                    identifiers={
                        "ClientInternal": lm.InstrumentIdValue(value=instrument_id)
                    },
                    look_through_portfolio_id=lm.ResourceId(scope=scope, code=portfolio),
                )
            }
        )
        print(f"Instrument {instr_result.values['look_through'].lusid_instrument_id} created")
    except ApiException as e:
        raise e


# 4. Book in Cash

Next, let's populate our portfolios with some cash. The cash will take one of three forms:

1. **Capital Commitments**
2. **Capital Calls**
3. **Expenses**

## Capital Commitments

A capital commitment is some projected capital expendature that an entity commits to spending over a set period of time. In our example, we use GBP.

In [ ]:
capital_commitments = pd.read_csv("./data/cash_commitments.csv")
capital_commitments

In [ ]:
# Create CapitalCommitment type
transaction_type_request = lm.TransactionTypeRequest(
    aliases=[
        lm.TransactionTypeAlias(
            type="CapitalCommitment", # Label
            description="Commitment of capital",
            transaction_class="CashTransfers",
            transaction_roles="Longer",
        )
    ], 
    movements=[
        # Buy side movement
        lm.TransactionTypeMovement(
            movement_types="CashReceivable", 
            side="Side1", # Buy properties
            direction=1, # Increase
        ),
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source="default",
        type="CapitalCommitment",
        transaction_type_request=transaction_type_request
    )
    print(f"Type '{response.aliases[0].type}' created")
    
except ApiException as e:
    raise e

Define our transaction mapping of fields that we'll use for upserts. This is for CapitalCommitments and we'll modify it for the other Transaction Types.

In [ ]:
transaction_mapping = {
    "identifier_mapping": {"Currency": "currency"},
    "required": {
        "code": "portfolio",
        "transaction_id": "txn_id",
        "type": "type",
        "transaction_price.price": "price",
        "transaction_price.type": "$Price",
        "total_consideration.amount": "total_consideration",
        "units": "quantity",
        "transaction_date": "trade_date",
        "total_consideration.currency": "currency",
        "settlement_date": "settlement_date",
    },
    "optional": {},
    "properties": ["investor", "holding_class"]
}

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=capital_commitments,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
)

if result['transactions']['errors']:
    # Throw error      
    raise result['transactions']['errors'][0]

succ, failed = format_transactions_response(result)
    
pd.DataFrame(
    data=[{"success": len(succ), "failed": len(failed)}]
)

## Capital Calls

Next, we can book capital calls for our general partnership. To do so, we are interacting directly with the securitised funds that we have just created.

In [ ]:
# Read our capital calls file into dataframe
capital_calls = pd.read_csv("./data/capital_calls.csv")

gp_calls = capital_calls[capital_calls["portfolio"] == "generalPartnership"]
display(gp_calls)


In [ ]:
# Create CapitalCall type
transaction_type_request = lm.TransactionTypeRequest(
    aliases=[
        lm.TransactionTypeAlias(
            type="CapitalCall", # Label
            description="Capital call",
            transaction_class="Basic",
            transaction_roles="LongLonger",
        )
    ], 
    movements=[
        # Buy side movement
        lm.TransactionTypeMovement(
            movement_types="StockMovement", 
            side="Side1", # Buy properties
            direction=1, # Increase
        ),
        # Sell side movement
        lm.TransactionTypeMovement(
            movement_types="CashCommitment",
            side="Side2", # Sell properties
            direction=-1, # Decrease
        )
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source="default",
        type="CapitalCall",
        transaction_type_request=transaction_type_request
    )
    print(f"Type '{response.aliases[0].type}' created")
    
except ApiException as e:
    raise e

In [ ]:
# Define the updated keys for Capital Call and update template dictionary
transaction_mapping_capital_call = {
    "identifier_mapping": {"ClientInternal": "client_id"},
    "properties": ["investor", "holding_class"]
}
transaction_mapping.update(transaction_mapping_capital_call)

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=gp_calls,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
)

if result['transactions']['errors']:
    # Throw error      
    raise result['transactions']['errors'][0]

succ, failed = format_transactions_response(result)
    
pd.DataFrame(
    data=[{"success": len(succ), "failed": len(failed)}]
)

In [ ]:
underlying_calls = capital_calls[capital_calls["portfolio"] != "generalPartnership"]
display(underlying_calls)

In [ ]:
# Define the updated keys for Underlying Calls and update template dictionary
transaction_mapping_underlying_calls = {
    "identifier_mapping": {"Currency": "currency"},
     "properties": ["holding_class"]
}
transaction_mapping.update(transaction_mapping_underlying_calls)

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=underlying_calls,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
)

if result['transactions']['errors']:
    # Throw error      
    raise result['transactions']['errors'][0]

succ, failed = format_transactions_response(result)
    
pd.DataFrame(
    data=[{"success": len(succ), "failed": len(failed)}]
)

## Expenses

To match something typical of the real-world, we can book some expenses that have accrued throughout various trading activities. Each expense will have a type, and those types will fall into one of three distinct holding classes. The structure of the expense classes is as follows:

1. **Management Fees**
    - Rebate 
2. **Partnership Expenses**
    - Accounting, Administration & IT
    - Audit & Tax Prepatory
    - Bank Fees
    - Custody Fees
    - Due Diligence
    - Legal
    - Organization Costs
    - Other Travel & Entertainment
    - Other
3. **Offsets**
    - Advisory Fee
    - Broken Deal Fee
    - Transaction & Deal Fee
    - Directors Fee
    - Monitoring Fee
    - Capital Markets Fee
    - Organization Costs
    - Placement Fee
    - Other

In [ ]:
expenses = pd.read_csv("./data/expenses.csv")
expenses.head()

In [ ]:
# Create our FundsIn and FundsOut Transaction Types

#FundsIn
transaction_type_request = lm.TransactionTypeRequest(
    aliases=[
        lm.TransactionTypeAlias(
            type="FundsIn", # Label
            description="Deposit New Funds",
            transaction_class="CashTransfers",
            transaction_roles="Longer",
        )
    ], 
    movements=[
        # Buy side movement
        lm.TransactionTypeMovement(
            movement_types="CashReceivable", 
            side="Side1", # Buy properties
            direction=1, # Increase
        )
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source="default",
        type="FundsIn",
        transaction_type_request=transaction_type_request
    )
    print(f"Type '{response.aliases[0].type}' created")
    
except ApiException as e:
    raise e

#FundsOut
transaction_type_request = lm.TransactionTypeRequest(
    aliases=[
        lm.TransactionTypeAlias(
            type="FundsOut", # Label
            description="Withdraw Funds",
            transaction_class="CashTransfers",
            transaction_roles="Shorter",
        )
    ], 
    movements=[
        # Buy side movement
        lm.TransactionTypeMovement(
            movement_types="CashReceivable", 
            side="Side1", # Buy properties
            direction=-1, # Decrease
        )
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source="default",
        type="FundsOut",
        transaction_type_request=transaction_type_request
    )
    print(f"Type '{response.aliases[0].type}' created")
    
except ApiException as e:
    raise e

In [ ]:
# Define the updated keys for Expenses and update template dictionary
transaction_mapping_expenses = {
    "properties": ["investor", "holding_class", "expense_type"]
}
transaction_mapping.update(transaction_mapping_expenses)

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=expenses,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
)

if result['transactions']['errors']:
    # Throw error      
    raise result['transactions']['errors'][0]

succ, failed = format_transactions_response(result)
    
pd.DataFrame(
    data=[{"success": len(succ), "failed": len(failed)}]
)

# 5. Load Data into Portfolios

In this section, we begin to populate the portfolios with transactions, quotes and cash flows. We must also instantiate our instruments.

## Private Equity Fund

In this section, we create the instruments and instrument properties for our Private Equity Fund.

The instrument categories are as follows, with each of these being represented by an equity instrument in LUSID:
1. **Preferred shares** (PFD)
2. **Simple Agreement for Future Equity** (SAFE)
3. **Equity**

In addition to these instruments, we create a dividend yield property. This is to be used with the preferred shares later on in the notebook.

In [ ]:
try:
    property_definition_request = lm.CreatePropertyDefinitionRequest(
        domain="Instrument",
        scope=scope,
        code="dividend_yield",
        display_name="Dividend Yield",
        data_type_id=lm.ResourceId(
            scope="system",
            code="number",
        ),
        life_time="Perpetual",
    )

    resp = property_definitions_api.create_property_definition(
        create_property_definition_request=property_definition_request
    )
    print(f"{resp.key} property created") 

except ApiException as e:
    if json.loads(e.body)["code"] == 124: # PropertyAlreadyExists
        print(json.loads(e.body)["title"])
    else:
        raise e

In [ ]:
def create_simple_instrument(
    name,
    type,
    client_internal,
    dom_ccy,
    dividend_yield,
):

    simple_instrument = lm.SimpleInstrument(
        instrument_type="SimpleInstrument",
        dom_ccy=dom_ccy,
        asset_class="Unknown",
        simple_instrument_type=type,
    )

    properties = lm.ModelProperty(
        key=f"Instrument/{scope}/dividend_yield",
        value=lm.PropertyValue(
            metric_value=lm.MetricValue(
                value=dividend_yield,
            )
        ),
    )

    instrument_definition = lm.InstrumentDefinition(
        name=name,
        identifiers={"ClientInternal": lm.InstrumentIdValue(value=client_internal)},
        definition=simple_instrument,
        properties=[properties],
    )

    # upsert the instrument
    upsert_request = {client_internal: instrument_definition}
    try:
        upsert_response = instruments_api.upsert_instruments(
            request_body=upsert_request)
        luid = upsert_response.values[client_internal].lusid_instrument_id
        return luid
    except ApiException as e:
        raise e

In [ ]:
instrument_type = "Equity"

name = "AECI LTD 5.5% - PFD"
identifier = "AECI_LTD_5.5"
dom_ccy = "GBP"
dividend_yield = 5.5

print(f"Instrument {create_simple_instrument(name, instrument_type, identifier, dom_ccy, dividend_yield)} created")

In [ ]:
name = "FINBOURNE Technology - SAFE"
identifier = "Finbourne_SAFE"
dom_ccy = "GBP"
dividend_yield = 0

print(f"Instrument {create_simple_instrument(name, instrument_type, identifier, dom_ccy, dividend_yield)} created")

In [ ]:
name = "Morrison Supermarkets PLC"
identifier = "Morrisons"
dom_ccy = "GBP"
dividend_yield = 0

print(f"Instrument {create_simple_instrument(name, instrument_type, identifier, dom_ccy, dividend_yield)} created")

## Real Estate Fund

Next, let's create the instruments and instrument properties for our Real Estate Fund.

The two real estate properties are:
1. **Sydney Bondi Junction Westfield**, a shopping centre in Australia
2. **One Carter Lane**, an office in London

The five properties we create are:
1. **Average Retail Lease**
2. **Term**
3. **Frequency**
4. **Rating**
5. **Rental Units**

In [ ]:
cmbs_properties = ['average_retail_lease', 'term', 'frequency', 'rating']

In [ ]:
try:
    property_definition_request = lm.CreatePropertyDefinitionRequest(
        domain="Instrument",
        scope=scope,
        code="rental_units",
        display_name="rental_units",
        data_type_id=lm.ResourceId(
            scope="system",
            code="number",
        ),
        life_time="Perpetual",
    )

    resp = property_definitions_api.create_property_definition(
        create_property_definition_request=property_definition_request
    )
    print(f"{resp.key} property created") 

except ApiException as e:
    if json.loads(e.body)["code"] == 124: # PropertyAlreadyExists
        print(json.loads(e.body)["title"])
    else:
        raise e

In [ ]:
for property in cmbs_properties:
    try:
        resp = property_definitions_api.create_property_definition(
            create_property_definition_request=lm.CreatePropertyDefinitionRequest(
                domain="Instrument",
                scope=scope,
                code=property,
                value_required=None,
                display_name=property,
                data_type_id=lm.ResourceId(scope="system", code="string"),
                life_time=None,
            )
        )
        print(f"{resp.key} property created") 
        
    except ApiException as e: 
        if json.loads(e.body)["code"] == 124: # PropertyAlreadyExists
            print(json.loads(e.body)["title"])
        else:
            raise e

In [ ]:
def create_cmbs_property(code, value):
    property = lm.ModelProperty(
                    key=f"Instrument/{scope}/{code}",
                    value=lm.PropertyValue(
                        label_value=value,
                    ),
                )
    return property

In [ ]:
def create_CMBS(
    name,
    client_internal,
    dom_ccy,
    rental_units,
    average_retail_lease,
    term,
    frequency,
    rating
):

    cmbs = lm.SimpleInstrument(
        instrument_type="SimpleInstrument",
        dom_ccy=dom_ccy,
        asset_class="Unknown",
        simple_instrument_type="CMBS"
    )

    # fill in dictionary of variables
    properties = []
    inputs = {
        "average_retail_lease": average_retail_lease,
        "term": term,
        "frequency": frequency,
        "rating": rating}

    # properties = lm.InstrumentProperties()
    properties.append(lm.ModelProperty(
        key=f"Instrument/{scope}/rental_units",
        value=lm.PropertyValue(
            metric_value=lm.MetricValue(
                value=rental_units,
            )
        ),
    ))

    for code in cmbs_properties:
        properties.append(create_cmbs_property(code, inputs[code]))


    cmbs_definition = lm.InstrumentDefinition(
        name=name,
        identifiers={"ClientInternal": lm.InstrumentIdValue(value=client_internal)},
        definition=cmbs,
        properties=properties,
    )

    # upsert the instrument
    upsert_request = {client_internal: cmbs_definition}
    try:
        upsert_response = instruments_api.upsert_instruments(request_body=upsert_request)
        cmbs_luid = upsert_response.values[client_internal].lusid_instrument_id
        return cmbs_luid
    except ApiException as e:
        raise e

In [ ]:
name="Sydney Bondi Junction Westfield"
client_internal="SYD241232"
dom_ccy="GBP"
rental_units=76
average_retail_lease="2.3Y"
term="5Y"
frequency="Q"
rating="B1"

sydney_luid = create_CMBS(name, client_internal, dom_ccy, rental_units, average_retail_lease, term, frequency, rating)
print(f"Instrument {sydney_luid} created")

In [ ]:
name="One Carter Lane"
client_internal="LON343239"
dom_ccy="GBP"
rental_units=50
average_retail_lease="1.8Y"
term="3Y"
frequency="SA"
rating="A1"

london_luid = create_CMBS(name, client_internal, dom_ccy, rental_units, average_retail_lease, term, frequency, rating)
print(f"Instrument {london_luid} created")

## Private Debt Fund

For our Private Debt Fund, we will create three instruments. These instruments will be one of two types:
1. **Bond**
    - Morrisons 4 05/01/2025
2. **Term Deposit**
    - HSBC 0.10% 05/01/2023
    - BNP 0.08% 05/07/2022

In [ ]:
def create_term_deposit(
    name,
    client_internal,
    start_date,
    maturity_date,
    contract_size,
    currency,
    payment_frequency,
    rate,
):

    term_deposit = lm.TermDeposit(
        start_date=start_date,
        maturity_date=maturity_date,
        contract_size=contract_size,
        flow_convention=lm.FlowConventions(
            currency=currency,
            payment_frequency=payment_frequency,
            roll_convention="MF",
            day_count_convention="Act365",
            payment_calendars=[],
            reset_calendars=[],
            settle_days=1,
            reset_days=0
            ),
        rate=rate,
        instrument_type="TermDeposit"
    )

    term_deposit_definition = lm.InstrumentDefinition(
        name=name,
        identifiers={"ClientInternal": lm.InstrumentIdValue(value=client_internal)},
        definition=term_deposit,
    )

    # upsert the instrument
    upsert_request = {client_internal: term_deposit_definition}
    try:
        upsert_response = instruments_api.upsert_instruments(request_body=upsert_request)
        term_deposit_luid = upsert_response.values[client_internal].lusid_instrument_id
        return term_deposit_luid
    except lu.ApiException as e:
        raise e

In [ ]:
def create_bond(
    name,
    client_internal,
    start_date,
    maturity_date,
    dom_ccy,
    coupon_rate,
    payment_frequency,
):

    bond = lm.Bond(
        start_date=start_date,
        maturity_date=maturity_date,
        dom_ccy=dom_ccy,
        principal=1,
        coupon_rate=coupon_rate,
        flow_conventions=lm.FlowConventions(
            scope=None,
            code=None,
            currency="GBP",
            payment_frequency=payment_frequency,
            roll_convention="None",
            day_count_convention="Act365",
            payment_calendars=[],
            reset_calendars=[],
            settle_days=2,
            reset_days=2
            ),
        identifiers={},
        instrument_type="Bond"
    )

    bond_definition = lm.InstrumentDefinition(
        name=name,
        identifiers={"ClientInternal": lm.InstrumentIdValue(value=client_internal)},
        definition=bond,
    )

    # upsert the instrument
    upsert_request = {client_internal: bond_definition}
    try:
        upsert_response = instruments_api.upsert_instruments(request_body=upsert_request)
        bond_luid = upsert_response.values[client_internal].lusid_instrument_id
        return bond_luid
    except lu.ApiException as e:
        raise e


In [ ]:
name="Morrisons 4 05/01/2025"
client_internal="Morrisons_4_05/01/2025_Private"
start_date=datetime(2022, 1, 5, 00, tzinfo=pytz.utc)
maturity_date=datetime(2025, 1, 5, 00, tzinfo=pytz.utc)
dom_ccy="GBP"
coupon_rate=0.04
payment_frequency="6M"

morrisons_luid = create_bond(
    name,
    client_internal,
    start_date,
    maturity_date,
    dom_ccy,
    coupon_rate,
    payment_frequency
)

print(f"Instrument {morrisons_luid} created")

In [ ]:
name = "HSBC 0.10% 05/01/2023"
client_internal = "HSBC_0.10_05/01/2023"
start_date = datetime(2022, 1, 5, 00, tzinfo=pytz.utc)
maturity_date = datetime(2023, 1, 5, 00, tzinfo=pytz.utc)
contract_size = 1
currency = "GBP"
payment_frequency = "6M"
rate = 0.001

hsbc_luid = create_term_deposit(
    name,
    client_internal,
    start_date,
    maturity_date,
    contract_size,
    currency,
    payment_frequency,
    rate,
)

print(f"Instrument {hsbc_luid} created")

In [ ]:
name = "BNP 0.08% 05/07/2022"
client_internal = "BNP_0.08_05/07/2022"
start_date = datetime(2022, 1, 5, 00, tzinfo=pytz.utc)
maturity_date = datetime(2022, 7, 5, 00, tzinfo=pytz.utc)
contract_size = 1
currency = "GBP"
payment_frequency = "6M"
rate = 0.0008

bnp_luid = create_term_deposit(
    name,
    client_internal,
    start_date,
    maturity_date,
    contract_size,
    currency,
    payment_frequency,
    rate,
)

print(f"Instrument {bnp_luid} created")

## Transactions

Next, we can populate our funds with transactions.

In [ ]:
transactions = pd.read_csv("./data/portfolio_transactions.csv")
display(transactions.head())

First, we'll create the Transaction Types referenced in the file, which are Buy and StockOut.

In [ ]:
# Buy
transaction_type_request = lm.TransactionTypeRequest(
    aliases=[
        lm.TransactionTypeAlias(
            type="Buy",  # Label
            description="Purchase",
            transaction_class="Basic",
            transaction_roles="LongLonger",
        )
    ],
    movements=[
        # Buy side movement
        lm.TransactionTypeMovement(
            movement_types="StockMovement",
            side="Side1",  # Buy properties
            direction=1,  # Increase
        ),
        # Sell side movement
        lm.TransactionTypeMovement(
            movement_types="CashCommitment",
            side="Side2",  # Sell properties
            direction=-1,  # Decrease
        ),
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source="default", type="Buy", transaction_type_request=transaction_type_request
    )
    print(f"Type '{response.aliases[0].type}' created")
except ApiException as e:
    raise e

# StockOut
transaction_type_request = lm.TransactionTypeRequest(
    aliases=[
        lm.TransactionTypeAlias(
            type="StockOut",  # Label
            description="Transfer Out",
            transaction_class="StockTransfers",
            transaction_roles="Shorter",
        )
    ],
    movements=[
        # Buy side movement
        lm.TransactionTypeMovement(
            movement_types="StockMovement",
            side="Side1",  # Sell properties
            direction=-1,  # Decrease
        )
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source="default",
        type="StockOut",
        transaction_type_request=transaction_type_request,
    )
    print(f"Type '{response.aliases[0].type}' created")
except ApiException as e:
    raise e

Next we'll upload the Private Equity and Private Debt fund transactions that do not have a Subholding Key. We'll re-use and update the same transaction mapping template variable from before and then load them into LUSID.

In [ ]:
private_debt_equity = transactions[(transactions["portfolio"] == "privateEquityFund") | (transactions["portfolio"] == "privateDebtFund")]
display(private_debt_equity)

# Define the updated keys for the real estate transactions
transaction_mapping_private_equity = {
    "identifier_mapping": {"ClientInternal": "client_id"},
    "properties": []
}
transaction_mapping.update(transaction_mapping_private_equity)

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=transactions,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
)

if result['transactions']['errors']:
    # Throw error
    raise result['transactions']['errors'][0]

succ, failed = format_transactions_response(result)

pd.DataFrame(
    data=[{"success": len(succ), "failed": len(failed)}]
)

Next we'll upsert Real Estate transactions with the Subholding Keys.

In [ ]:
real_estate = transactions[transactions["portfolio"] == "realEstateFund"]
display(real_estate)

transaction_mapping_holding_class= {
    "identifier_mapping": {"ClientInternal": "client_id"},
    "properties": ["holding_class"]
}
transaction_mapping.update(transaction_mapping_holding_class)

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=real_estate,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
)

if result['transactions']['errors']:
    # Throw error      
    raise result['transactions']['errors'][0]

succ, failed = format_transactions_response(result)
    
pd.DataFrame(
    data=[{"success": len(succ), "failed": len(failed)}]
)

## Quotes

We can add quotes which represent the value of our entities at specific points in time. This allows them to be valued by the valuations engine.

In [ ]:
quotes = pd.read_csv("./data/quotes.csv")
quotes.head()

In [ ]:
quotes_mapping = {
    "quote_id.quote_series_id.instrument_id_type": "$ClientInternal",
    "quote_id.effective_at": "date",
    "quote_id.quote_series_id.provider": "$Lusid",
    "quote_id.quote_series_id.quote_type": "$Price",
    "quote_id.quote_series_id.instrument_id": "client_id",
    "metric_value.unit": "currency",
    "quote_id.quote_series_id.var_field": "$mid",
    "metric_value.value": "quote",
}

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=quotes,
    mapping_required=quotes_mapping,
    mapping_optional={},
    file_type="quotes",
)

if result["quotes"]["errors"]:
    # Throw error
    raise result["quotes"]["errors"][0]

succ, failed, errors = format_quotes_response(result)

display(
    pd.DataFrame(
        data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}]
    )
)

## Recipes

In order to create valuations we require a recipe. In this case, we are creating two. For an overview of recipe structure and creation see [here](https://support.lusid.com/knowledgebase/article/KA-01895), and for information regarding lookthrough recipes for use in fund of funds portfolios see [here](https://support.lusid.com/knowledgebase/article/KA-01854/en-us).

The first recipe is a **lookthrough** recipe which means that it will fetch the valuations of the securitised funds during a valuation of the General Partnership (parent portfolio). 

The second recipe is a **non-lookthrough** recipe. This does not look into our securitised portfolios, and instead will value them by using quotes in our quotes store.

In [ ]:
# Create look-through-enabled recipe
lookthrough_config_recipe = lm.ConfigurationRecipe(
    scope=scope,
    code="lookthrough",
    market=lm.MarketContext(
        # The Market Rules tell the recipe how to look up to pricing data for different instrument types in order of priority
        market_rules=[
            # Look in the LUSID Quote Store for mid prices loaded using ClientInternal identifier, going back 10Y if needed
            lm.MarketDataKeyRule(
                key="Equity.ClientInternal.*",
                supplier="Lusid",
                data_scope=scope,
                quote_type="Price",
                field="mid",
                quote_interval="10Y"
            ),
            # Look in the LUSID Complex Market Data store for any Credit Curves, going back 10Y if needed
            lm.MarketDataKeyRule(
                key="Credit.*.*",
                supplier="Lusid",
                data_scope=scope,
                quote_type="Price",
                quote_interval="10Y"
            ),
        ],
        # We can specify default suppliers based on asset class as a fallback
        suppliers=lm.MarketContextSuppliers(
            commodity="Lusid",
            credit="Lusid",
            equity="Lusid",
            fx="Lusid",
            rates="Lusid"
        ),
        # Configure options on the recipe, such as the default supplier, default instrument identifier type and to infer missing FX if only inverse pair is available
        options=lm.MarketOptions(
            default_supplier="Lusid",
            default_instrument_code_type="ClientInternal",
            default_scope=scope,
            attempt_to_infer_missing_fx=True
        ),
    ),
    # We must define the pricing model to use when valuing our positions
    pricing=lm.PricingContext(
        model_rules=[
            # Here we specify that for SimpleInstruments we want to value using built-in SimpleStatic model
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="SimpleStatic",
                instrument_type="SimpleInstrument",
                # The IndexModelOptions allows us to 'look through' to value holdings in child portfolios
                model_options=lm.IndexModelOptions(
                    portfolio_scaling="Unity",
                    model_options_type="IndexModelOptions"
                ),
            ),
        ],
        options=lm.PricingOptions(
            window_valuation_on_instrument_start_end=False
        ),
        # This is how we access our external cashflows from the Structured Result Store (which we'll load later on in this notebook)
        result_data_rules=[
            lm.ResultDataKeyRule(
                resource_key="UnitResult/*",
                supplier="Client",
                data_scope=scope,
                document_code="realEstateCashFlows",
                quote_interval="10Y",
                document_result_type="UnitResult/Analytic",
                result_key_rule_type="ResultDataKeyRule"
            )
        ],
    ),
)

try:
    response = configuration_recipe_api.upsert_configuration_recipe(
        upsert_recipe_request=lm.UpsertRecipeRequest(
            configuration_recipe=lookthrough_config_recipe
        )
    )
    print(f"Successful recipe creation at {response.value}")
except ApiException as e:
    raise e

In [ ]:
# Create a non-look-through recipe
non_lookthrough_config_recipe = lm.ConfigurationRecipe(
    scope=scope,
    code="no-lookthrough",
    market=lm.MarketContext(
        # The Market Rules tell the recipe how to look up to pricing data for different instrument types in order of priority
        market_rules=[
            # Look in the LUSID Quote Store for mid prices loaded using ClientInternal identifier
            lm.MarketDataKeyRule(
                key="Equity.ClientInternal.*",
                supplier="Lusid",
                data_scope=scope,
                quote_type="Price",
                field="mid",
            ),
            # Look into the LUSID Quote Store for any applicable FX currency pair rates
            lm.MarketDataKeyRule(
                key="FX.CurrencyPair.*",
                supplier="Lusid",
                data_scope=scope,
                quote_type="Rate",
                field="mid",
            ),
        ],
        suppliers=lm.MarketContextSuppliers(
            commodity="Lusid",
            credit="Lusid",
            equity="Lusid",
            fx="Lusid",
            rates="Lusid"
        ),
        options=lm.MarketOptions(
            default_supplier="Lusid",
            default_instrument_code_type="ClientInternal",
            default_scope=scope,
            attempt_to_infer_missing_fx=True,
        ),
    ),
    pricing=lm.PricingContext(
        # Note how the model rule here does NOT specify an IndexModelOption to look through to child funds
        model_rules=[
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="IndexPrice",
                instrument_type="Index",
            )
        ],
        options=lm.PricingOptions(
            window_valuation_on_instrument_start_end=False
        ),
        # Note how this recipe does not have a result_data_rules parameter, so no overrides will take place on valuations that use it
    ),
)

configuration_recipe_api.upsert_configuration_recipe(
    upsert_recipe_request=lm.UpsertRecipeRequest(
        configuration_recipe=non_lookthrough_config_recipe
    )
)

try:
    response = configuration_recipe_api.upsert_configuration_recipe(
        upsert_recipe_request=lm.UpsertRecipeRequest(
            configuration_recipe=lookthrough_config_recipe
        )
    )
    print(f"Successful recipe creation at {response.value}")
except ApiException as e:
    raise e

# 6. Excel ILPA Report Writer (for General Partnership)

This section will demonstrate how we are able to take data from LUSID, perform some data organisation, and produce an excel report. We will be following the ILPA template, a popular format for reporting in the private equity industry. See [here](https://ilpa.org/reporting-template/) for more info.

The outputted template will show and aggregation of 3 investors and the expenses and cash accrued for each.

In [ ]:
def get_ilpa_data(investors, from_date, to_date):
    # get transactions for investors
    transactions = transaction_portfolios_api.get_transactions(
        scope=scope,
        code="generalPartnership",
        from_transaction_date=from_date,
        to_transaction_date=to_date,
    ).values
    
    # parse response
    transaction_df = pd.DataFrame(
        data={
            "ID": [i.transaction_id for i in transactions],
            "Units": [i.units for i in transactions],
            "Type": [i.type for i in transactions],
            "Holding Class": [i.properties['Transaction/privateMarkets/holding_class'].value.label_value if 'Transaction/privateMarkets/holding_class' in i.properties else 'None' for i in transactions],
            "Investor": [i.properties['Transaction/privateMarkets/investor'].value.label_value if 'Transaction/privateMarkets/investor' in i.properties else 'None' for i in transactions],
            "Expense Type": [i.properties['Transaction/privateMarkets/expense_type'].value.label_value if 'Transaction/privateMarkets/expense_type' in i.properties else 'None' for i in transactions]
        }
    )

    ilpa_temp = {
        'Total Offsets to Fees & Expenses (applied during period)': 0,
        'Management Fees – Gross of Offsets, Waivers & Rebates': 0,
        'Contributions - Cash & Non-Cash': 0,
        'Distributions - Cash & Non-Cash (input positive values)': 0
    }
    
    for investor in investors:
        for index, row in transaction_df.iterrows():
            if row['Investor'] == investor:
                # define our number of units
                units = row['Units']
                # negative movement
                if row['Type'] == 'FundsOut':
                    units = -units
                    
                # Partnership Expenses
                if row['Holding Class'] == 'Partnership Expense':
                    key = f"Partnership Expenses - {row['Expense Type']}"
                # Management Fee    
                elif row['Holding Class'] == 'Management Fee':
                    key = f"Management Fee {row['Expense Type']}"
                    # add to total
                    ilpa_temp['Management Fees – Gross of Offsets, Waivers & Rebates'] += units
                # Offsets
                elif row['Holding Class'] == 'Offset':
                    key = f"{row['Expense Type']} Offset"
                    # add to total
                    ilpa_temp['Total Offsets to Fees & Expenses (applied during period)'] += units
                    ilpa_temp['Management Fees – Gross of Offsets, Waivers & Rebates'] += units
                # Cash
                elif row['Holding Class'] == 'Cash':
                    key = "Beginning NAV - Net of Incentive Allocation"
                elif row['Holding Class'] == 'Fund' and row['Type'] == 'Buy':
                    # add to total
                    ilpa_temp['Contributions - Cash & Non-Cash'] += units
                elif row['Holding Class'] == 'Fund' and row['Type'] == 'Sell':
                    # add to total
                    ilpa_temp['Distributions - Cash & Non-Cash (input positive values)'] += units
                else:
                    continue

                # add to our data template
                if key not in ilpa_temp:
                    ilpa_temp[key] = units
                else:
                    ilpa_temp[key] += units
    
    return ilpa_temp
            

Build the data into ILPA format

In [ ]:
def build_ilpa_template(lp, tf, gp, qtd, ytd, si, end):
    ilpa_data = {}
    
    # Format Dates
    qtd_str = datetime.fromisoformat(qtd).strftime("%B %d, %Y")
    ytd_str = datetime.fromisoformat(ytd).strftime("%B %d, %Y")
    si_str = datetime.fromisoformat(si).strftime("%B %d, %Y")
    end_str = datetime.fromisoformat(end).strftime("%B %d, %Y")
    
    # Limited Partner
    lp_qtd = get_ilpa_data(lp, qtd, end)
    lp_ytd = get_ilpa_data(lp, ytd, end)
    lp_si = get_ilpa_data(lp, si, end)
    
    # Total Fund
    tf_qtd = get_ilpa_data(tf, qtd, end)
    tf_ytd = get_ilpa_data(tf, ytd, end)
    tf_si = get_ilpa_data(tf, si, end)
    
    # General Partner
    gp_qtd = get_ilpa_data(gp, qtd, end)
    gp_ytd = get_ilpa_data(gp, ytd, end)
    gp_si = get_ilpa_data(gp, si, end)
    
    # Combine all data
    for dictionary in [lp_qtd, lp_ytd, lp_si, tf_qtd, tf_ytd, tf_si, gp_qtd, gp_ytd, gp_si]:
        for key in dictionary:
            if key in ilpa_data:
                ilpa_data[key].append(dictionary[key])
            else:
                ilpa_data[key] = [dictionary[key]]

    ilpa_template = pd.DataFrame(
        data=ilpa_data,
        index=[f"LP QTD ({qtd_str} - {end_str})", 
               f"LP YTD ({ytd_str} - {end_str})", 
               f"LP Since Inception ({si_str} - {end_str})",
               f"Total Fund QTD ({qtd_str} - {end_str})", 
               f"Total Fund YTD ({ytd_str} - {end_str})", 
               f"Total Fund Since Inception ({si_str} - {end_str})",
               f"GP QTD ({qtd_str} - {end_str})", 
               f"GP YTD ({ytd_str} - {end_str})", 
               f"GP Since Inception ({si_str} - {end_str})"]
    )
    
    ilpa_template = ilpa_template.swapaxes("index", "columns")
    return ilpa_template

# Partners
lp = ["Investor 1"]
tf = ["Investor 1", "Investor 2", "Investor 3"]
gp = ["Investor 3"]

# Dates for search
qtd = "2022-04-01T00:00:00+00:00"
ytd = "2022-01-01T00:00:00+00:00"
si = "2021-01-01T00:00:00+00:00"
end = "2022-06-15T00:00:00+00:00" 

ilpa_template = build_ilpa_template(lp, tf, gp, qtd, ytd, si, end)
ilpa_template.head()

Create the ILPA excel report. You can view this output within the `outputs` folder with filename `ilpa_report.xlsx`.

In [ ]:
# Load blank ILPA template
workbook = openpyxl.load_workbook("./data/ILPA-template.xlsx")
worksheet = workbook.active

# Load excel cell mapping from JSON
with open("./data/ilpa_mapping.json", "r") as fp:
    excel_cells = json.load(fp)

# Add values
for index, row in ilpa_template.iterrows():
    for i in range(len(row)):
        cell = excel_cells[index][i]
        worksheet[cell].value = row[i]

# Add dates
for index, date in enumerate([qtd, ytd, si, end]):
    date_str = datetime.fromisoformat(date).strftime("%d/%m/%Y")
    cell = excel_cells["Dates"][index]
    worksheet[cell].value = date_str

os.makedirs("./outputs", exist_ok=True)
response = workbook.save("./outputs/ilpa_report.xlsx")

# Response is None if succesful
if not response:
    print("File saved at ./outputs/ilpa_report.xlsx")
else:
    print(response)

# 7. Real Estate Cash Flows

In this section we will further adapt our real estate portfolio by including some realistic cash flows. The cash flows will represent rental income/outgoings and bond coupon payments.

## Override Cash Flows

Our real estate assets are modelled as Simple Instruments, and as such, they **do not** natively have cash flows on their instruments. However, we are able to store cashflows from external sources, such as a third party, against any instrument by adding them into our [Structured Results Store (SRS)](https://support.lusid.com/knowledgebase/article/KA-01893/). During a valuation, by using a specific recipe configuration we are able to instruct the valuations engine to fetch the cash flows we have added.

In [ ]:
cash_flow_df = pd.read_csv("./data/cash_flow.csv")

cash_flow_df["settlement_date"] = pd.to_datetime(cash_flow_df["settlement_date"], format='%d/%m/%Y', utc=True)

In [ ]:
document_scope = "privateMarkets"
document_code = "realEstateCashFlows"
effective_at = datetime(2022, 1, 1, 0, 0, tzinfo=pytz.utc)


def override_cashflow(luid, cash_flows, notional):
    # Persist pay amounts - maintenance costs
    cash_flow = [
        lm.CashFlowValue(
            result_value_type="CashFlowValue",
            payment_amount=row["total_consideration"] / notional,
            payment_date=row["settlement_date"],
            payment_ccy=row["currency"],
            cash_flow_lineage=lm.CashFlowLineage(
                cash_flow_type="Cash", pay_receive=row["pay_receive"]
            ),
        )
        for i, row in cash_flows.iterrows()
    ]

    # Create external cash flow set from pays and receives
    external_cash_flow_set = lm.CashFlowValueSet(
        cashflows=cash_flow, result_value_type="CashFlowValueSet"
    )

    # Create a structured document ID
    struct_result_data_id = lm.StructuredResultDataId(
        source="Client",
        code=document_code,
        effective_at=effective_at.isoformat(),
        result_type="UnitResult/Analytic",
    )

    upsert_result_values_data_request = lm.UpsertResultValuesDataRequest(
        document_id=struct_result_data_id,
        key={
            # Instrument to which document resolves
            f"UnitResult/LusidInstrumentId": f"{luid}"
        },
        data_address="UnitResult/Valuation/Cashflows",  # Valuation result to override
        result_value=external_cash_flow_set,
    )

    try:
        response = structured_result_data_api.upsert_result_value(
            scope=document_scope,
            request_body={"CashFlows": upsert_result_values_data_request},
        )
        if not response.failed:
            return f"All cash flows uploaded successfully for {luid}"
    except ApiException as e:
        display(json.loads(e.body)["status"], json.loads(e.body)["title"])


# Call function providing filtered cashflows for London
print(
    override_cashflow(
        london_luid, cash_flow_df[cash_flow_df.client_id == "LON343239"], 197823
    )
)
# Call function providing filtered cashflows for Sydney
print(
    override_cashflow(
        sydney_luid, cash_flow_df[cash_flow_df.client_id == "SYD241232"], 199400
    )
)

## Book Bond

Unlike Simple Instruments, Bond instruments **do** support cashflows natively. By providing an economic definition for the bond, the LUSID valuations engine is able to project cash flows through to maturity.

By booking a bond with an economic definition into this portfolio, we will be able to see how our valuations engine is able to dynamically integrate both the native instrument cash flows and the overriden SRS cash flows.

In [ ]:
scope = "privateMarkets"
portfolio_code = "realEstateFund"
trade_date = datetime(2022, 1, 4, tzinfo=pytz.utc)
start_date = datetime(2022, 1, 4, tzinfo=pytz.utc)
maturity_date = datetime(2030, 1, 1, tzinfo=pytz.utc)
bond_id = 300

# Create Bond Instrument
flow_conventions = lm.FlowConventions(
    currency="GBP",
    payment_frequency="1M",
    roll_convention="ModifiedPrevious",
    day_count_convention="Actual365",
    payment_calendars=[],
    reset_calendars=[],
    settle_days=2,
    reset_days=2,
)

bond = lm.Bond(
    start_date=start_date,
    maturity_date=maturity_date,
    dom_ccy="GBP",
    principal=1,
    coupon_rate=0.05,
    flow_conventions=flow_conventions,
    identifiers={},
    instrument_type="Bond",
    calculation_type="Standard",
)

# Define the instrument to be upserted
bond_definition = lm.InstrumentDefinition(
    name="Fixed Bond",
    identifiers={"ClientInternal": lm.InstrumentIdValue(value="Fixed Bond")},
    definition=bond,
    properties=[],
)

# Upsert the instrument
upsert_request = {"Fixed Bond": bond_definition}
try:
    upsert_response = instruments_api.upsert_instruments(
        request_body=upsert_request, scope="default"
    )
    luid = upsert_response.values["Fixed Bond"].lusid_instrument_id
    print(f"Bond {luid} successfully created")
except ApiException as e:
    raise e

# Create Transaction Type for sell
transaction_type_request = lm.TransactionTypeRequest(
    aliases=[
        lm.TransactionTypeAlias(
            type="Sell",  # Label
            description="Sale",
            transaction_class="Basic",
            transaction_roles="LongShorter",
        )
    ],
    movements=[
        # Sell side movement
        lm.TransactionTypeMovement(
            movement_types="StockMovement",
            side="Side1",  # Sell properties
            direction=-1,  # Decrease
        ),
        # Buy side movement
        lm.TransactionTypeMovement(
            movement_types="CashCommitment",
            side="Side2",  # Buy properties
            direction=1,  # Increase
        ),
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source="default", type="Sell", transaction_type_request=transaction_type_request
    )
    print(f"Type '{response.aliases[0].type}' created")
except ApiException as e:
    raise e

# Create Transaction
trade_txn = lm.TransactionRequest(
    transaction_id=f"tx{bond_id}",
    type="Sell",
    instrument_identifiers={"Instrument/default/LusidInstrumentId": luid},
    transaction_date=trade_date.isoformat(),
    settlement_date=trade_date.isoformat(),
    units=20000000,
    transaction_price=lm.TransactionPrice(price=1, type="Price"),
    total_consideration=lm.CurrencyAndAmount(amount=1 * 20000000, currency="GBP"),
    exchange_rate=1,
    source=scope,
    transaction_currency="GBP",
    properties={
        "Transaction/privateMarkets/holding_class": lm.PerpetualProperty(
            key="Transaction/privateMarkets/holding_class",
            value=lm.PropertyValue(label_value="Bond"),
        )
    },
)

try:
    response = transaction_portfolios_api.upsert_transactions(
        scope=scope, code=portfolio_code, transaction_request=[trade_txn]
    )

    print(f"Transaction successfully updated at time: {response.version.as_at_date}")
except ApiException as e:
    raise e

### Get Cash Ladder

By fetching a cash ladder, we can see how the valuations engine is able to combine the two types of cash flows into one cohesive projected cash statement.

In [ ]:
response = transaction_portfolios_api.get_portfolio_cash_ladder(
    scope="privateMarkets",
    code="realEstateFund",
    from_effective_at="2022-01-01T00:00:00+00:00",
    to_effective_at="2032-01-01T19:20:30+00:00",
    effective_at="2023-01-01T00:00:00+00:00",
    recipe_id_scope = "privateMarkets",
    recipe_id_code = "lookthrough"
)



cl_data = {
    "Instrument": [],
    "Date": [],
    "Activity": [],
    "Amount": [],
    "Open": [],
    "Close":[]
}

# each value
for val in response.values:
    name = val.sub_holding_keys['Transaction/privateMarkets/holding_class'].value.label_value 
    # each record
    for i in val.records:
        # each activity in record
        for key in i.activities:
            cl_data["Instrument"].append(name)
            cl_data["Date"].append(i.effective_date)
            cl_data["Activity"].append(key)
            cl_data["Amount"].append(i.activities[key])
            cl_data["Open"].append(i.open)
            cl_data["Close"].append(i.close)

pd.DataFrame(data=cl_data).set_index("Date").sort_index().head(10)

## Data Calculations

Using the cash flows, we will now perform some basic data processing and data calculations to help understand what our data means.

### Request Cash Flows

First, request the real estate cash flows from LUSID.

In [ ]:
end_date = datetime(2032, 1, 1, 0, 0, tzinfo=pytz.utc)
effective_at = datetime(2022, 1, 11, 0, 0, tzinfo=pytz.utc)

# Request portfolio cash flows
external_cash_flow_response = transaction_portfolios_api.get_portfolio_cash_flows(
    scope="privateMarkets",
    code="realEstateFund",
    effective_at=effective_at.isoformat(),
    window_start=effective_at.isoformat(),
    window_end=end_date.isoformat(),
    recipe_id_scope="privateMarkets",
    recipe_id_code="lookthrough"
)

cf_data = {
    'Payment Date':[],
    'One Carter Lane':[],
    'Sydney Bondi Junction Westfield':[]
}

for val in external_cash_flow_response.values:
    # amount positive if recieve, negative if pay
    amount = val.amount

    if val.source_instrument_id == london_luid:
        cf_data['One Carter Lane'].append(amount)
        cf_data['Payment Date'].append(val.payment_date)

    elif val.source_instrument_id == sydney_luid:
        cf_data['Sydney Bondi Junction Westfield'].append(amount)

cash_flow_df = pd.DataFrame(data=cf_data)
cash_flow_df['Payment Date'] = pd.to_datetime(cash_flow_df['Payment Date'])
cash_flow_df.set_index('Payment Date', inplace=True)

# Plot net cash flows for year 1
date_cash_flow_df = cash_flow_df
date_cash_flow_df.index = date_cash_flow_df.index.date

cash_flow_df.head()

### Group by Yearly Cash Flow

In [ ]:
cash_flow_yearly = cash_flow_df.groupby(lambda x: x.year).sum()
cash_flow_yearly

### Internal Rate of Return (IRR)

The annual rate of growth that an investment is expected to generate, over the 10 years we have cash flow data for.

In [ ]:
# Basic IRR algorithm
def npv(rate, cashflows):
    total = 0.0
    for i, cashflow in enumerate(cashflows):
        total += cashflow / (1 + rate)**i
    return total

def irr(cashflows, iterations=100):
    if len(cashflows)==0:
        raise 'Error - number of cash flows cannot be zero'
    rate = 1.0
    investment = cashflows[0]
    for i in range(1, iterations+1):
        rate *= (1 - npv(rate, cashflows) / investment)
    # as 2dp percentage value
    return '{:.2%}'.format(rate)

# Get from Real Estate data and parse
london_intial = float(real_estate[real_estate.client_id == 'LON343239']['total_consideration'].item().replace(',', ''))
sydney_intial = float(real_estate[real_estate.client_id == 'SYD241232']['total_consideration'].item().replace(',', ''))

london_irr = irr([-london_intial] + list(cash_flow_yearly['One Carter Lane']))
sydney_irr = irr([-sydney_intial] + list(cash_flow_yearly['Sydney Bondi Junction Westfield']))

print(f'{london_irr} - One Carter Lane internal rate of return (IRR)')
print(f'{sydney_irr} - Sydney Bondi Junction Westfield internal rate of return (IRR)')

### Return on Investment (ROI)

Next, we can calculate how long it will take until our intial investment is paid off.

In [ ]:
cumulative_cash_flows = cash_flow_df.cumsum()
cumulative_cash_flows.head()

In [ ]:
# Get earliest date where cash is > intial cost
actual_london_roi_df = cumulative_cash_flows[cumulative_cash_flows['One Carter Lane'] > london_intial].idxmin()
actual_london_roi = actual_london_roi_df['One Carter Lane']

actual_sydney_roi_df = cumulative_cash_flows[cumulative_cash_flows['Sydney Bondi Junction Westfield'] > sydney_intial].idxmin()
actual_sydney_roi = actual_sydney_roi_df['Sydney Bondi Junction Westfield']

print(f"{actual_london_roi} One Carter Lane ROI acheived")
print(f"{actual_sydney_roi} Sydney Bondi Junction Westfield ROI acheived")

In [ ]:
def actual_roi(end_date, client_id):
    
    buy_date = real_estate[real_estate.client_id == client_id]['trade_date'].item()
    buy_datetime = datetime.strptime(buy_date, "%Y-%m-%dT%H:%M:%SZ")

    roi_delta = relativedelta(end_date, buy_datetime)
    
    return f"{roi_delta.years} years, {roi_delta.months} months and {roi_delta.days} days"

london_actual_roi = actual_roi(actual_london_roi, 'LON343239')
sydney_actual_roi = actual_roi(actual_sydney_roi, 'SYD241232')

print(f"{london_actual_roi} to achieve ROI for One Carter Lane")
print(f"{sydney_actual_roi} to achieve ROI for Sydney Bondi Junction Westfield")


In [ ]:
pdf_data = pd.DataFrame(data={
    "Type": [
        "IRR",
        "Date of ROI",
        "Length of ROI"
    ],
    "One Carter Lane":[
        london_irr, 
        actual_london_roi.strftime("%m/%d/%Y, %H:%M:%S"),
        london_actual_roi
    ],
    "Sydney Bondi Junction Westfield": [
        sydney_irr, 
        actual_sydney_roi.strftime("%m/%d/%Y, %H:%M:%S"), 
        sydney_actual_roi
    ]
})

pdf_data

## Data Visualisation

We can begin to visualise our cash flows on both a short term and long term basis.

In [ ]:
ax = date_cash_flow_df.head(24).plot.bar(y=['One Carter Lane', 'Sydney Bondi Junction Westfield'], color=['#ff5200', '#0000ff'], figsize=[15, 10])

plt.title('Cash Flow Chart - Year 1')
plt.ylabel('Cash Amount (GBP)')
plt.xlabel('Date')

ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))

# Save the graph
plt.savefig('./outputs/cash_flow_y1.png')

In [ ]:
ax = cumulative_cash_flows.plot.line(
    y=['One Carter Lane', 'Sydney Bondi Junction Westfield'], 
    color=['#ff5200', '#0000ff'], 
    figsize=[9, 6]
)

plt.title('Cumulative Cash Flow')
plt.ylabel('Cash Amount (USD)')
plt.xlabel('Date')

ax.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))
ax.xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y'))

plt.axhline(london_intial, color='r')
plt.axhline(sydney_intial, color='r')

ax.annotate(
    'Initial Cost of Properties',
    xy=('2022-01-31 00:00:00+00:00', 19000000), 
    xytext=('2022-01-31 00:00:00+00:00', 19000000)
)

# Save the graph
plt.savefig('./outputs/cumulative_cash_flow.png')

# 8. PDF Report Writer (for Real Estate Fund)

Using the calculations, cash flows and graphs from above, we are able to combine them to produce a concise and informative PDF report.

In [ ]:
pdf = FPDF()

# Set cell height
ch = 8

# Title and Logo
pdf.add_page()
pdf.set_font("Arial", style='B', size=14)
pdf.text(txt='Real Estate Report', x=10, y=10)
pdf.image("./data/finbourne.png", x=150, y=3, w=50)
pdf.ln(10)

# Calculations Table header
pdf.set_font("Arial", style='B', size=10)
pdf.set_fill_color(235, 235, 235)
pdf.cell(w=30, h=ch, txt='Type', border=1, ln=0, align='C', fill=True)
pdf.cell(w=80, h=ch, txt='One Carter Lane (OCL)', border=1, ln=0, align='C', fill=True)
pdf.cell(w=80, h=ch, txt='Sydney Bondi Junction Westfield (SBJW)', border=1, ln=1, align='C', fill=True)

# Calculations Table contents
pdf.set_font('Arial', '', 10)
for i in range(0, len(pdf_data)):
    pdf.cell(w=30, h=ch, 
             txt=pdf_data['Type'].iloc[i], 
             border=1, ln=0, align='C')
    pdf.cell(w=80, h=ch, 
             txt=pdf_data['One Carter Lane'].iloc[i], 
             border=1, ln=0, align='C')
    pdf.cell(w=80, h=ch, 
             txt=pdf_data['Sydney Bondi Junction Westfield'].iloc[i], 
             border=1, ln=1, align='C')
pdf.ln(ch)

# Yearly Cashflow header
pdf.set_font("Arial", style='B', size=10)
pdf.set_fill_color(235, 235, 235) # grey
pdf.cell(w=20, h=ch, txt='Year', border=1, ln=0, align='C', fill=True)
pdf.cell(w=20, h=ch, txt='OCL', border=1, ln=0, align='C', fill=True)
pdf.cell(w=20, h=ch, txt='SBJW', border=1, ln=1, align='C', fill=True)

# Yearly Cashflow contents
pdf.set_font('Arial', '', 10)
for i in range(0, len(cash_flow_yearly)):
    pdf.cell(w=20, h=ch, 
             txt=str(cash_flow_yearly.index[i]),
             border=1, ln=0, align='C')
    pdf.cell(w=20, h=ch, 
             txt=str(round(cash_flow_yearly['One Carter Lane'].iloc[i])), 
             border=1, ln=0, align='C')
    pdf.cell(w=20, h=ch, 
             txt=str(round(cash_flow_yearly['Sydney Bondi Junction Westfield'].iloc[i])), 
             border=1, ln=1, align='C')

# Add Cumulative Cash Flow graph
pdf.image("./outputs/cumulative_cash_flow.png", x=75, y=58, w=138)

# Add Yearly Cash Flow graph
pdf.image("./outputs/cash_flow_y1.png", y=140, w=222, x=-5)

# Save PDF
pdf.output("./outputs/real_estate_report.pdf", "F")

# Display PDF
IFrame("./outputs/real_estate_report.pdf", width=800, height=1200)